In [ ]:
import sys
from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
import pytorch_lightning as pl

REPO_ROOT = next(
    path
    for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "common" / "paths.py").is_file()
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from common.paths import ROOT

from common.dataset import make_loader
from common.metrics import calculate_fid, dump_json, image_metric_frame, load_weight_only, save_weight_only, summarize_image_metrics
from common.models import PatchDiscriminator, TranslationNetwork, initialize_map2sat
from common.paths import PRETRAINED_GENERATOR
from common.split import assign_patient_split
from common.train import BestStateCallback, EpochRecorder

DATASET_NAME = "kidney"
EXPERIMENT_NAME = "external_pretraining_ablation"
NOTEBOOK_DIR = ROOT / "experiment" / "kidney" / "04_other"
BASE_PAIR_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "pairs_all.csv"
EXTERNAL_PAIR_CSV = ROOT / "experiment" / "kidney" / "01_data_processing" / "cache" / "external_pairs.csv"
training = False
SEEDS = [42, 43, 44, 45, 46]
PRETRAINING_OPTIONS = {
    "without_pretrained": False,
    "with_pretrained": True,
}
BATCH_SIZE = 8
MAX_EPOCHS = 80
NUM_WORKERS = 4
LEARNING_RATE = 2e-4

In [ ]:
class TranslationLightning(pl.LightningModule):
    def __init__(self, use_fem, learning_rate=2e-4, reconstruction_weight=100.0):
        super().__init__()
        self.automatic_optimization = False
        self.network = TranslationNetwork(use_fem=use_fem)
        self.discriminator = PatchDiscriminator()
        self.learning_rate = learning_rate
        self.reconstruction_weight = reconstruction_weight
        self.adversarial_loss = nn.BCEWithLogitsLoss()
        self.reconstruction_loss = nn.L1Loss()

    def forward(self, inputs):
        return self.network(inputs)

    def training_step(self, batch, batch_index):
        generator_optimizer, discriminator_optimizer = self.optimizers()
        gray = batch["gray"]
        target = batch["swe"]
        generated = self(gray)
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        real_logits = self.discriminator(gray, target)
        fake_logits = self.discriminator(gray, generated.detach())
        discriminator_loss = 0.5 * (
            self.adversarial_loss(real_logits, torch.ones_like(real_logits))
            + self.adversarial_loss(fake_logits, torch.zeros_like(fake_logits))
        )
        discriminator_optimizer.zero_grad()
        self.manual_backward(discriminator_loss)
        discriminator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(False)
        fake_logits = self.discriminator(gray, generated)
        adversarial = self.adversarial_loss(fake_logits, torch.ones_like(fake_logits))
        reconstruction = self.reconstruction_loss(generated, target)
        generator_loss = adversarial + self.reconstruction_weight * reconstruction
        generator_optimizer.zero_grad()
        self.manual_backward(generator_loss)
        generator_optimizer.step()
        for parameter in self.discriminator.parameters():
            parameter.requires_grad_(True)
        self.log("train_generator_loss", generator_loss, prog_bar=True)
        self.log("train_discriminator_loss", discriminator_loss, prog_bar=True)
        self.log("train_reconstruction", reconstruction)
        return generator_loss

    def validation_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("val_reconstruction", loss, prog_bar=True, on_epoch=True)
        return loss

    def test_step(self, batch, batch_index):
        generated = self(batch["gray"])
        loss = self.reconstruction_loss(generated, batch["swe"])
        self.log("test_reconstruction", loss, on_epoch=True)
        return loss

    def predict_step(self, batch, batch_index, dataloader_index=0):
        return {
            "gray": batch["gray"].detach().cpu(),
            "prediction": self(batch["gray"]).detach().cpu(),
            "target": batch["swe"].detach().cpu(),
            "emean": batch["emean"].detach().cpu(),
            "image_name": list(batch["image_name"]),
            "patient_id": list(batch["patient_id"]),
        }

    def configure_optimizers(self):
        generator_optimizer = torch.optim.Adam(
            self.network.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        discriminator_optimizer = torch.optim.Adam(
            self.discriminator.parameters(),
            lr=self.learning_rate,
            betas=(0.5, 0.999),
        )
        return [generator_optimizer, discriminator_optimizer]

In [ ]:
def collect_translation(outputs):
    predictions = torch.cat([output["prediction"] for output in outputs])
    targets = torch.cat([output["target"] for output in outputs])
    image_names = [name for output in outputs for name in output["image_name"]]
    patient_ids = [name for output in outputs for name in output["patient_id"]]
    return predictions, targets, image_names, patient_ids


def evaluate_translation(model, trainer, loader, output_prefix):
    outputs = trainer.predict(model, loader)
    predictions, targets, image_names, patient_ids = collect_translation(outputs)
    image_frame = image_metric_frame(predictions, targets, image_names, patient_ids)
    summary, patient_frame = summarize_image_metrics(image_frame)
    summary["fid"] = calculate_fid(
        predictions,
        targets,
        NOTEBOOK_DIR / f"fid_{output_prefix}",
        "cuda:0" if torch.cuda.is_available() else "cpu",
    )
    image_frame.to_csv(
        NOTEBOOK_DIR / f"{output_prefix}_image_metrics.csv",
        index=False,
    )
    patient_frame.to_csv(
        NOTEBOOK_DIR / f"{output_prefix}_patient_image_metrics.csv",
        index=False,
    )
    return summary

In [ ]:
base_frame = pd.read_csv(BASE_PAIR_CSV).drop(columns=["split"])
external_frame = pd.read_csv(EXTERNAL_PAIR_CSV)
rows = []
for seed in SEEDS:
    frame = assign_patient_split(base_frame, seed=seed)
    train_frame = frame.loc[frame["split"] == "train"].copy()
    validation_frame = frame.loc[frame["split"] == "val"].copy()
    test_frame = frame.loc[frame["split"] == "test"].copy()
    train_loader = make_loader(train_frame, BATCH_SIZE, True, True, NUM_WORKERS)
    validation_loader = make_loader(validation_frame, BATCH_SIZE, False, False, NUM_WORKERS)
    test_loader = make_loader(test_frame, BATCH_SIZE, False, False, NUM_WORKERS)
    external_loader = make_loader(external_frame, BATCH_SIZE, False, False, NUM_WORKERS)
    for option_name, use_pretrained in PRETRAINING_OPTIONS.items():
        pl.seed_everything(seed, workers=True)
        weight_path = NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_{option_name}_seed_{seed}_generator.pt"
        model = TranslationLightning(False, LEARNING_RATE)
        if training and use_pretrained:
            initialize_map2sat(model.network.generator, PRETRAINED_GENERATOR)
        best = BestStateCallback("val_reconstruction", "network")
        trainer = pl.Trainer(
            max_epochs=MAX_EPOCHS,
            accelerator="auto",
            devices=1,
            precision="16-mixed" if torch.cuda.is_available() else "32-true",
            deterministic=True,
            logger=False,
            enable_checkpointing=False,
            enable_model_summary=False,
            enable_progress_bar=False,
            callbacks=[
                EpochRecorder(NOTEBOOK_DIR / f"{EXPERIMENT_NAME}_{option_name}_seed_{seed}_epoch_metrics.csv"),
                best,
                pl.callbacks.EarlyStopping(
                    monitor="val_reconstruction",
                    mode="min",
                    patience=15,
                ),
            ],
            num_sanity_val_steps=0,
        )
        if training:
            trainer.fit(model, train_loader, validation_loader)
            save_weight_only(model.network, weight_path)
            reloaded_network = TranslationNetwork(False)
            load_weight_only(reloaded_network, weight_path)
            model.network.load_state_dict(reloaded_network.state_dict())
        else:
            if not weight_path.is_file():
                raise FileNotFoundError(weight_path)
            load_weight_only(model.network, weight_path)
        trainer.validate(model, validation_loader)
        trainer.test(model, test_loader)
        internal_summary = evaluate_translation(
            model,
            trainer,
            test_loader,
            f"{EXPERIMENT_NAME}_{option_name}_seed_{seed}_internal",
        )
        trainer.test(model, external_loader)
        external_summary = evaluate_translation(
            model,
            trainer,
            external_loader,
            f"{EXPERIMENT_NAME}_{option_name}_seed_{seed}_external",
        )
        row = {
            "seed": seed,
            "option": option_name,
            "use_pretrained": use_pretrained,
            "internal_fid": internal_summary["fid"],
            "external_fid": external_summary["fid"],
        }
        for cohort_name, summary in (
            ("internal", internal_summary),
            ("external", external_summary),
        ):
            for level in ("image_level", "patient_level"):
                for metric, statistics in summary[level].items():
                    row[f"{cohort_name}_{level}_{metric}"] = statistics["mean"]
        rows.append(row)

result_frame = pd.DataFrame(rows)
result_frame.to_csv(
    NOTEBOOK_DIR / "external_pretraining_ablation_by_seed.csv",
    index=False,
)
summary_rows = []
for option_name, option_frame in result_frame.groupby("option", sort=False):
    for column in result_frame.columns:
        if column in {"seed", "option", "use_pretrained"}:
            continue
        values = pd.to_numeric(option_frame[column], errors="coerce").dropna()
        summary_rows.append(
            {
                "option": option_name,
                "metric": column,
                "mean": float(values.mean()),
                "sd": float(values.std(ddof=1)) if len(values) > 1 else 0.0,
            }
        )
summary_frame = pd.DataFrame(summary_rows)
summary_frame.to_csv(
    NOTEBOOK_DIR / "external_pretraining_ablation_mean_sd.csv",
    index=False,
)
dump_json(
    {
        "dataset": DATASET_NAME,
        "experiment": EXPERIMENT_NAME,
        "results": summary_rows,
    },
    NOTEBOOK_DIR / "metrics_external_pretraining_ablation.json",
)
display(summary_frame)